## Working with TiTiler-EoPF


In [ ]:
# Start titiler-eopf services locally with Docker

!docker compose up api -d

In [ ]:
import json
import httpx2 as httpx
from folium import Map, TileLayer

from IPython.display import Image

%matplotlib inline

In [ ]:
titiler_endpoint = "http://127.0.0.1:8000"

### Conformances

In [ ]:
r = httpx.get(f"{titiler_endpoint}/conformance").json()
print(json.dumps(r, indent=4))

In [ ]:
collection_id = "sentinel-2-l2a"
item_id = "S2C_MSIL2A_20260903T135731_N0512_R010_T27WVV_20260903T171314"

In [ ]:
# List all available assets for the given item
r = httpx.get(
    f"{titiler_endpoint}/collections/{collection_id}/items/{item_id}/assets",
    timeout=20,
).json()

print("list of available assets: ")
print(json.dumps(list(r), indent=4))

## Variables

In [ ]:
# Fetch Metadata for all available variables
r = httpx.get(
    f"{titiler_endpoint}/collections/{collection_id}/items/{item_id}/assets/reflectance/dataset/keys",
    timeout=20,
).json()

print("list of supported variables: ")
print(json.dumps(list(r), indent=4))

### Info

In [ ]:
# Fetch Metadata for all available variables
r = httpx.get(
    f"{titiler_endpoint}/collections/{collection_id}/items/{item_id}/assets/reflectance/info",
    timeout=10,
).json()

print(json.dumps(r, indent=4))

In [ ]:
# Fetch Metadata for a specific Variable
r = httpx.get(
    f"{titiler_endpoint}/collections/{collection_id}/items/{item_id}/assets/reflectance/info",
    params={
        "variables": "b04",
    },
    timeout=10,
).json()

print(json.dumps(r, indent=4))

## Display tiles

### Single Variable

In [ ]:
r = httpx.get(
    f"{titiler_endpoint}/collections/{collection_id}/items/{item_id}/assets/reflectance/WebMercatorQuad/tilejson.json",
    params={
        "variables": "b04",
        "rescale": "0,0.3",
        "tilesize": 256,
    },
    timeout=10,
).json()
print(r)
bounds = r["bounds"]
m = Map(
    location=((bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2), zoom_start=8
)

TileLayer(tiles=r["tiles"][0], opacity=1, attr="ESA EoPF").add_to(m)
m

### Band Combination: False Color Nir/Red/Green

In [ ]:
r = httpx.get(
    f"{titiler_endpoint}/collections/{collection_id}/items/{item_id}/assets/reflectance/WebMercatorQuad/tilejson.json",
    params=[
        ("variables", "b04"),
        ("variables", "b03"),
        ("variables", "b02"),
        ("rescale", "0,0.3"),
        ("tilesize", 256),
    ],
).json()
print(r)
bounds = r["bounds"]
m = Map(
    location=((bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2), zoom_start=9
)

TileLayer(tiles=r["tiles"][0], opacity=1, attr="ESA EoPF").add_to(m)

m

### Band Math: NDVI

In [ ]:
red = "b04"
nir = "b8a"

r = httpx.get(
    f"{titiler_endpoint}/collections/{collection_id}/items/{item_id}/assets/reflectance/WebMercatorQuad/tilejson.json",
    params=[
        ("expression", f"({nir}-{red})/({nir}+{red})"),
        ("rescale", "-1,1"),
        ("colormap_name", "viridis"),
        ("tilesize", 256),
    ],
    timeout=10,
).json()
print(r)
bounds = r["bounds"]
m = Map(
    location=((bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2), zoom_start=9
)

TileLayer(tiles=r["tiles"][0], opacity=1, attr="ESA EoPF").add_to(m)

m

### Preview

In [ ]:
r = httpx.get(
    f"{titiler_endpoint}/collections/{collection_id}/items/{item_id}/assets/reflectance/preview.png",
    params=[
        ("variables", "b04"),
        ("variables", "b03"),
        ("variables", "b02"),
        ("rescale", "0,0.3"),
        ("max_size", 256),
    ],
)
print(r.headers)
Image(r.content)

### OGC Maps

In [ ]:
r = httpx.get(
    f"{titiler_endpoint}/collections/{collection_id}/items/{item_id}/assets/reflectance/map",
    params=[
        ("variables", "b04"),
        ("variables", "b03"),
        ("variables", "b02"),
        ("rescale", "0,0.3"),
        ("width", 256),
        ("height", 256),
        ("f", "png"),
    ],
)
print(r.headers)
Image(r.content)

In [ ]:
r = httpx.get(
    f"{titiler_endpoint}/collections/{collection_id}/items/{item_id}/assets/reflectance/map",
    params=[
        ("variables", "b04"),
        ("variables", "b03"),
        ("variables", "b02"),
        ("rescale", "0,0.3"),
        ("width", 256),
        ("height", 256),
        ("f", "png"),
        ("crs", "EPSG:32633"),
    ],
)
print(r.headers)
Image(r.content)

### Part

In [ ]:
r = httpx.get(
    f"{titiler_endpoint}/collections/{collection_id}/items/{item_id}/assets/reflectance/bbox/-23.264684,71.515392,-21.825561,72.113103.png",
    params=[
        ("variables", "b04"),
        ("variables", "b03"),
        ("variables", "b02"),
        ("rescale", "0,0.3"),
        ("max_size", 1024),
    ],
)
print(r.headers)
Image(r.content)

In [ ]:
feat = {
    "type": "Feature",
    "properties": {},
    "geometry": {
        "type": "Polygon",
        "coordinates": [
            [
                [-23.264683754829832, 72.11310261918777],
                [-21.82556070720935, 72.11310261918777],
                [-21.82556070720935, 71.51539186859],
                [-23.264683754829832, 71.51539186859],
                [-23.264683754829832, 72.11310261918777],
            ]
        ],
    },
}


r = httpx.post(
    f"{titiler_endpoint}/collections/{collection_id}/items/{item_id}/assets/reflectance/feature.png",
    json=feat,
    params=[
        ("variables", "b04"),
        ("variables", "b03"),
        ("variables", "b02"),
        ("rescale", "0,0.3"),
        ("max_size", 1024),
    ],
)
print(r.headers)
Image(r.content)